# Prediction Request ke Heart Disease Model – Flask + TensorFlow

Notebook ini digunakan untuk menguji dan melakukan prediction request ke sistem machine learning yang telah dijalankan di cloud menggunakan **Flask + TensorFlow**.

**Model:** Heart Disease Classification  
**Serving URL:** `https://heart-disease-mlops-production.up.railway.app`  
**Endpoint predict:** `/predict`  
**Endpoint health:** `/`  
**Endpoint metrics:** `/metrics`  
**Dataset:** Heart Disease UCI  

## 1. Import Library

In [ ]:
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

print('Semua library berhasil diimport!')

## 2. Konfigurasi URL

In [ ]:
SERVING_URL  = 'https://heart-disease-mlops-production.up.railway.app'

HEALTH_URL   = f'{SERVING_URL}/'
PREDICT_URL  = f'{SERVING_URL}/predict'
METRICS_URL  = f'{SERVING_URL}/metrics'
INFO_URL     = f'{SERVING_URL}/model/info'

print(f'Health endpoint   : {HEALTH_URL}')
print(f'Predict endpoint  : {PREDICT_URL}')
print(f'Metrics endpoint  : {METRICS_URL}')

## 3. Health Check – Cek Status Model

In [ ]:
resp = requests.get(HEALTH_URL, timeout=10)
print('Status HTTP :', resp.status_code)
print(json.dumps(resp.json(), indent=2))

## 4. Helper – Fungsi Prediksi

Model sekarang di-serve via **Flask + TensorFlow** langsung.  
Input cukup berupa JSON biasa dengan nilai fitur pasien — tidak perlu serialisasi tf.Example.

In [ ]:
def predict_single(patient: dict) -> dict:
    """Kirim satu pasien ke Flask API dan kembalikan hasil prediksi."""
    resp = requests.post(
        PREDICT_URL,
        json=patient,
        headers={'Content-Type': 'application/json'},
        timeout=15
    )
    resp.raise_for_status()
    return resp.json()


def batch_predict(patients_df: pd.DataFrame) -> pd.DataFrame:
    """Prediksi batch untuk seluruh baris DataFrame."""
    results = []
    for _, row in patients_df.iterrows():
        patient = {k: v for k, v in row.to_dict().items() if k != 'target'}
        results.append(predict_single(patient))

    out = patients_df.copy()
    out['probability']     = [r['confidence']           for r in results]
    out['predicted_label'] = [r['prediction']           for r in results]
    out['confidence']      = [r['confidence']           for r in results]
    out['diagnosis']       = [r['diagnosis']            for r in results]
    return out


print('Helper functions siap digunakan!')

## 5. Single Prediction – Contoh Pasien

In [ ]:
# Pasien dengan risiko penyakit jantung (target sebenarnya = 1)
patient_positive = {
    'age': 63, 'sex': 1, 'cp': 3, 'trestbps': 145, 'chol': 233,
    'fbs': 1, 'restecg': 0, 'thalach': 150, 'exang': 0,
    'oldpeak': 2.3, 'slope': 0, 'ca': 0, 'thal': 1
}

print('Data Pasien:')
print(json.dumps(patient_positive, indent=2))
print()

result = predict_single(patient_positive)
print('Hasil Prediksi:')
print(json.dumps(result, indent=2))

In [ ]:
# Pasien tanpa penyakit jantung (target sebenarnya = 0)
patient_negative = {
    'age': 67, 'sex': 1, 'cp': 0, 'trestbps': 160, 'chol': 286,
    'fbs': 0, 'restecg': 0, 'thalach': 108, 'exang': 1,
    'oldpeak': 1.5, 'slope': 1, 'ca': 3, 'thal': 2
}

print('Data Pasien:')
print(json.dumps(patient_negative, indent=2))
print()

result = predict_single(patient_negative)
print('Hasil Prediksi:')
print(json.dumps(result, indent=2))

## 6. Batch Prediction dari Dataset

In [ ]:
# Muat dataset
df = pd.read_csv('josa_pratama-pipeline/data/raw/heart.csv')
print(f'Shape dataset: {df.shape}')
display(df.head())

# Ambil 10 sampel untuk testing
test_samples = df.sample(10, random_state=42).reset_index(drop=True)
print(f'\nMelakukan prediksi untuk {len(test_samples)} sampel...')

In [ ]:
# Batch prediction
results_df = batch_predict(test_samples)

# Tampilkan hasil
display_cols = ['age', 'sex', 'target', 'predicted_label', 'probability', 'confidence', 'diagnosis']
display(results_df[display_cols])

# Hitung akurasi
correct = (results_df['target'] == results_df['predicted_label']).sum()
total   = len(results_df)
print(f'\nAkurasi pada sampel: {correct}/{total} = {correct/total:.1%}')

## 7. Visualisasi Hasil Prediksi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribusi probability score
axes[0].hist(
    results_df[results_df['target'] == 1]['probability'].dropna(),
    bins=10, alpha=0.7, label='Actual Positive (Heart Disease)', color='red'
)
axes[0].hist(
    results_df[results_df['target'] == 0]['probability'].dropna(),
    bins=10, alpha=0.7, label='Actual Negative (No Disease)', color='green'
)
axes[0].axvline(x=0.5, color='black', linestyle='--', label='Threshold = 0.5')
axes[0].set_title('Distribusi Probability Score')
axes[0].set_xlabel('Probability')
axes[0].set_ylabel('Count')
axes[0].legend()

# Plot 2: Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(
    results_df['target'],
    results_df['predicted_label'].astype(int)
)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
    xticklabels=['No Disease', 'Disease'],
    yticklabels=['No Disease', 'Disease']
)
axes[1].set_title('Confusion Matrix (10 Sampel)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('prediction_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan sebagai prediction_results.png')

## 8. Latency Benchmark

In [ ]:
import time

n_requests = 20
latencies  = []

for i in range(n_requests):
    start = time.time()
    predict_single(patient_positive)
    latencies.append((time.time() - start) * 1000)

print(f'Benchmark Results ({n_requests} requests):')
print(f'  Min latency : {min(latencies):.1f} ms')
print(f'  Max latency : {max(latencies):.1f} ms')
print(f'  Avg latency : {np.mean(latencies):.1f} ms')
print(f'  P95 latency : {np.percentile(latencies, 95):.1f} ms')
print(f'  Throughput  : {1000/np.mean(latencies):.1f} req/sec')

## 9. Cek Prometheus Metrics

In [ ]:
resp = requests.get(METRICS_URL, timeout=10)
print(f'Status: {resp.status_code}')
print('\nMetrik (sample):')
for line in resp.text.split('\n'):
    if 'heart_disease' in line and not line.startswith('#'):
        print(line)